In [22]:
import pandas as pd
import json
from pandas import json_normalize
from rich.theme import Theme
from rich.console import Console

from tqdm import tqdm  #shows progress bar ( can see that the code is running)
import time 


for i in tqdm(range(100), desc = "result"):
    time.sleep(0.1)

custom_theme = Theme({'error': "bold red", 'warning': "bold yellow", 'info': "bold blue", 'success': "bold green"})

console = Console(theme=custom_theme)

# Load the data
movies = pd.read_csv(r'/Users/utkugulbardak/Downloads/movies.csv')

# Convert the 'Awards' column from a string representation of JSON to a Python list of dictionaries
movies['Awards'] = movies['Awards'].apply(lambda x: json.loads(x) if pd.notnull(x) and isinstance(x, str) else [])

# Repeat the 'name' column for each award entry (this step is crucial for your case)
movies_expanded = movies.explode('Awards', ignore_index=True)

# Now, 'name' will be correctly repeated for each award
movies_expanded['Name'] = movies_expanded['Name'].ffill()

# Normalize the 'Awards' column into individual columns for each award
awards_data = json_normalize(movies_expanded['Awards'])

# Make sure missing columns are filled with NaN
required_columns = ['awardName', 'categoryName', 'eventName', 'isCompany', 'isPerson', 'isPrimary', 'isSecondary', 'isTitle', 'isWinner', 'year']
for column in required_columns:
    if column not in awards_data.columns:
        awards_data[column] = pd.NA  # fill missing columns with NaN

# Combine the original movie data with the exploded and normalized awards data
result = pd.concat([movies_expanded.drop(columns=['Awards']), awards_data], axis=1)

# Save the cleaned and flattened data to a new CSV file
result.to_csv(r'/Users/utkugulbardak/Downloads/cleaned_data.csv', index=False)



result: 100%|██████████| 100/100 [00:10<00:00,  9.41it/s]


In [21]:
moviescsv = pd.read_csv(r'/Users/utkugulbardak/Downloads/cleaned_data.csv')

moviescsv.head()

,Name,awardName,categoryName,eventName,isCompany,isPerson,isPrimary,isSecondary,isTitle,isWinner,year
0,David Niven,BAFTA Film Award,Best British Actor,BAFTA Awards,False,True,True,False,False,False,1955
1,David Niven,Special Award,NaN,Evening Standard British Film Awards,False,True,True,False,False,True,1981
2,David Niven,Golden Apple,Male Star of the Year,Golden Apple Awards,False,True,True,False,False,False,1975
3,David Niven,Sant Jordi,Best Foreign Actor (Mejor Actor Extranjero),Sant Jordi Awards,False,True,True,False,False,True,1960
4,David Niven,Oscar,Best Actor in a Leading Role,"Academy Awards, USA",False,True,True,False,False,True,1959


In [16]:
moviescleaned = moviescsv.drop(columns =['isPerson', 'isCompany', 'isPrimary', 'isSecondary', 'isTitle'])




In [17]:
display(moviescleaned)


,Name,awardName,categoryName,eventName,isWinner,year
0,David Niven,BAFTA Film Award,Best British Actor,BAFTA Awards,False,1955
1,David Niven,Special Award,NaN,Evening Standard British Film Awards,True,1981
2,David Niven,Golden Apple,Male Star of the Year,Golden Apple Awards,False,1975
3,David Niven,Sant Jordi,Best Foreign Actor (Mejor Actor Extranjero),Sant Jordi Awards,True,1960
4,David Niven,Oscar,Best Actor in a Leading Role,"Academy Awards, USA",True,1959
...,...,...,...,...,...,...
64209,Asta Paredes,Home Media Magazine Award,Best Scream Queen,Home Media Magazine Awards,False,2014
64210,Asta Paredes,July Prize,Honorable Mention,Women's Only Entertainment Film Festival,True,2017
64211,Asta Paredes,December Award,Best Actress -Short Film,Queen Palm International Film Festival,False,2018
64212,Asta Paredes,Festival Prize,Best Actress,Fright Night Theatre Film Festival (FNTFF),True,2014


In [18]:
result.to_csv(r'/Users/utkugulbardak/Downloads/moviescleaned.csv', index=False)

In [19]:
#check where the awardName is equal to 'Oscar' at the same year and category
 
oscar = moviescleaned[moviescleaned['awardName'] == 'Oscar']

oscar_same = oscar[oscar.duplicated(subset = ['year', 'categoryName'], keep = False)]

oscar.head()

,Name,awardName,categoryName,eventName,isWinner,year
4,David Niven,Oscar,Best Actor in a Leading Role,"Academy Awards, USA",True,1959
184,Ian McKellen,Oscar,Best Actor in a Leading Role,"Academy Awards, USA",False,1999
198,Ian McKellen,Oscar,Best Actor in a Supporting Role,"Academy Awards, USA",False,2002
533,Francis Lai,Oscar,"Best Music, Original Score","Academy Awards, USA",True,1971
551,Bradley Cooper,Oscar,Best Adapted Screenplay,"Academy Awards, USA",False,2019
